In [8]:
import os
import warnings
import pickle
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
DATA_PATH       = "/kaggle/input/datasets/quanghuylt/combined/combined.csv"
METRICS_DIR     = "./reports/metrics/arimax/"
META_TRAIN_DIR  = "./data/meta_train/"
META_TEST_DIR   = "./data/meta_test/"
FIGURES_DIR     = "./reports/figures/"
MODEL_DIR       = "./models/arimax/"

SYMBOLS   = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
EXOG_COLS = ["dxy", "fed_rate", "gold", "oil", "sp500"]

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

In [ ]:
def split_data(df: pd.DataFrame):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    
    n = len(df)
    train_idx = int(n * TRAIN_RATIO)
    val_idx   = train_idx + int(n * VAL_RATIO)
    
    train_df  = df.iloc[:train_idx].reset_index(drop=True)
    val_df    = df.iloc[train_idx:val_idx].reset_index(drop=True)
    test_df   = df.iloc[val_idx:].reset_index(drop=True)
    
    return train_df, val_df, test_df

def prepare_exog(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[EXOG_COLS] = df[EXOG_COLS].diff()
    df = df.dropna().reset_index(drop=True)
    return df

In [ ]:
def find_best_order(endog: pd.Series, exog: pd.DataFrame) -> tuple:
    log.info("    Running auto_arima to find best (p,d,q)...")
    model = auto_arima(
        endog,
        exogenous=exog,
        seasonal=False,          
        test="adf", 
        d=None,
        stepwise=True,
        information_criterion="aic",
        error_action="ignore",
        suppress_warnings=True,
        trace=False,
        max_p=3, max_q=3,
    )
    order = model.order
    log.info(f"    Best order={order}")
    return order

def predict_set(final_result, eval_df: pd.DataFrame, ticker: str, set_name: str) -> pd.DataFrame:
    result_applied = final_result.apply(
        eval_df["Close"],
        exog=eval_df[EXOG_COLS],
        refit=False
    )

    return pd.DataFrame({
        "Ticker": ticker,
        "Set": set_name,
        "Date": eval_df["Date"].values,
        "Close": eval_df["Close"].values,
        "ARIMAX_Predicted_Close": result_applied.fittedvalues.values,
    })

def calc_metrics(df: pd.DataFrame, set_name: str = "Test") -> pd.DataFrame:
    df = df.copy()
    df = df.dropna(subset=["Close", "ARIMAX_Predicted_Close"])
    
    df["Prev_Actual"] = df["Close"].shift(1)
    df["Prev_Pred"]   = df["ARIMAX_Predicted_Close"].shift(1)
    df = df.dropna(subset=["Prev_Actual", "Prev_Pred"])

    y_true = df["Close"]
    y_pred = df["ARIMAX_Predicted_Close"]

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / y_true + 1e-8))
    r2   = r2_score(y_true, y_pred)

    actual_dir = np.sign(y_true.values - df["Prev_Actual"].values)
    pred_dir   = np.sign(y_pred.values - df["Prev_Pred"].values)
    da         = np.mean(actual_dir == pred_dir)

    mask = actual_dir != 0
    tpa  = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan

    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "Set":    set_name,
        "MSE":    mse,
        "MAE":    mae,
        "MAPE":   mape,
        "RMSE":   rmse,
        "R2":     r2,
        "DA":     da,
        "TPA":    tpa,
        "V-RMSE": v_rmse,
    }])

In [ ]:
def plot_results(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, ticker: str, save_dir: str):
    os.makedirs(save_dir, exist_ok=True)
    sns.set(style="whitegrid")

    fig, axes = plt.subplots(2, 1, figsize=(15, 10), gridspec_kw={'height_ratios': [3, 1]})

    eval_actual_df = pd.concat([val_df, test_df], ignore_index=True)

    axes[0].plot(eval_actual_df["Date"], eval_actual_df["Close"], 
                 label="Actual Close", color="#1f77b4", linewidth=2)
    
    axes[0].plot(val_df["Date"], val_df["ARIMAX_Predicted_Close"], 
                 label="Val Predicted", color="#ff7f0e", linestyle="--", linewidth=1.5)
    
    axes[0].plot(test_df["Date"], test_df["ARIMAX_Predicted_Close"], 
                 label="Test Predicted", color="#d62728", linestyle="--", linewidth=1.5)

    axes[0].set_title(f"ARIMAX Predicted Close: {ticker} (Val & Test)", fontsize=15)
    axes[0].set_ylabel("Price", fontsize=12)
    axes[0].legend(loc="upper right")
    axes[0].grid(True, alpha=0.5)

    error = eval_actual_df["Close"] - eval_actual_df["ARIMAX_Predicted_Close"]
    
    colors = np.where(error >= 0, "#2ca02c", "#d62728")
    axes[1].bar(eval_actual_df["Date"], error, color=colors, alpha=0.4, width=1.5)
    axes[1].axhline(0, color="black", linewidth=1, linestyle="--")

    axes[1].set_title(f"Prediction Error: {ticker}", fontsize=13)
    axes[1].set_xlabel("Date", fontsize=12)
    axes[1].set_ylabel("Error (Actual - Predicted)", fontsize=12)
    axes[1].grid(True, alpha=0.5)

    for ax in axes:
        for label in ax.get_xticklabels():
            label.set_rotation(30)
            label.set_horizontalalignment('right')

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"arimax_{ticker}.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()
    log.info(f"    Plot saved: {save_path}")

In [ ]:
def train_one_ticker(ticker: str) -> tuple:
    log.info(f"========== {ticker} ==========")

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)

    train_df, val_df, test_df = split_data(df)

    train_df = prepare_exog(train_df)
    val_df   = prepare_exog(val_df)
    test_df  = prepare_exog(test_df)

    log.info(f"    Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} rows")

    order = find_best_order(train_df["Close"], train_df[EXOG_COLS])
    
    log.info("    Training ARIMAX on 80% Train Set...")
    model_fit = ARIMA(
        train_df["Close"],
        exog=train_df[EXOG_COLS],
        order=order,
    ).fit()

    val_result_df = predict_set(model_fit, val_df, ticker, "Validation")
    val_metrics   = calc_metrics(val_result_df, "Validation")
    val_metrics.insert(0, "Ticker", ticker)
    
    log.info("    Re-training ARIMAX on 90% (Train + Val) for Test Set...")
    train_val_df = pd.concat([train_df, val_df], ignore_index=True)
    final_model = ARIMA(
        train_val_df["Close"].values, 
        exog=train_val_df[EXOG_COLS].values, 
        order=order
    ).fit()
    
    test_result_df = predict_set(final_model, test_df, ticker, "Test")
    test_metrics   = calc_metrics(test_result_df, "Test")
    test_metrics.insert(0, "Ticker", ticker)
    
    log.info(f"    Val  — MAE={val_metrics['MAE'].iloc[0]:.2f} DA={val_metrics['DA'].iloc[0]:.2%} R2={val_metrics['R2'].iloc[0]:.4f}")
    log.info(f"    Test — MAE={test_metrics['MAE'].iloc[0]:.2f} DA={test_metrics['DA'].iloc[0]:.2%} R2={test_metrics['R2'].iloc[0]:.4f}")

    os.makedirs(META_TRAIN_DIR, exist_ok=True)
    os.makedirs(META_TEST_DIR, exist_ok=True)
    
    val_result_df.to_csv(os.path.join(META_TRAIN_DIR, f"ARIMAX_{ticker}_val_predictions.csv"), index=False)
    test_result_df.to_csv(os.path.join(META_TEST_DIR, f"ARIMAX_{ticker}_test_predictions.csv"), index=False)

    plot_results(train_df, val_result_df, test_result_df, ticker, save_dir=FIGURES_DIR)

    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, f"arimax_{ticker}.pkl"), "wb") as f:
        pickle.dump({
            "result": final_model,
            "order": order,
            "ticker": ticker,
        }, f)

    log.info(f"    All outputs saved for {ticker}")
    
    return val_metrics, test_metrics

In [ ]:
all_val_metrics = []
all_test_metrics = []

for ticker in SYMBOLS:
    try:
        val_m, test_m = train_one_ticker(ticker)
        all_val_metrics.append(val_m)
        all_test_metrics.append(test_m)
    except Exception as e:
        log.error(f"{ticker} FAILED: {e}")

if all_val_metrics:
    val_summary_df = pd.concat(all_val_metrics, ignore_index=True)
    os.makedirs(METRICS_DIR, exist_ok=True)
    VAL_METRICS_PATH = os.path.join(METRICS_DIR, "ARIMAX_all_tickers_val_metrics.csv")
    val_summary_df.to_csv(VAL_METRICS_PATH, index=False)
    log.info(f"\nValidation Summary saved: {VAL_METRICS_PATH}")
    print("\n--- VALIDATION METRICS ---")
    print(val_summary_df.to_string(index=False))

if all_test_metrics:
    test_summary_df = pd.concat(all_test_metrics, ignore_index=True)
    os.makedirs(METRICS_DIR, exist_ok=True)
    TEST_METRICS_PATH = os.path.join(METRICS_DIR, "ARIMAX_all_tickers_test_metrics.csv")
    test_summary_df.to_csv(TEST_METRICS_PATH, index=False)
    log.info(f"\nTest Summary saved: {TEST_METRICS_PATH}")
    print("\n--- TEST METRICS ---")
    print(test_summary_df.to_string(index=False))

log.info("Done.")

13:08:20  INFO      ========== VNM ==========
13:08:20  INFO          Train: 3258 | Val: 406 | Test: 407 rows
13:08:20  INFO          Running auto_arima to find best (p,d,q)...
13:08:26  INFO          Best order=(0, 1, 0)
13:08:26  INFO          Training ARIMAX on 80% Train Set...
13:08:27  INFO          Re-training ARIMAX on 90% (Train + Val) for Test Set...
13:08:30  INFO          Val  — MAE=0.53 DA=44.44% R2=0.9464
13:08:30  INFO          Test — MAE=0.69 DA=43.10% R2=0.9447
13:08:31  INFO          Plot saved: ./reports/figures/arimax_VNM.png
13:08:31  INFO          All outputs saved for VNM
13:08:31  INFO      ========== FPT ==========
13:08:31  INFO          Train: 3258 | Val: 406 | Test: 407 rows
13:08:31  INFO          Running auto_arima to find best (p,d,q)...
13:08:42  INFO          Best order=(1, 1, 3)
13:08:42  INFO          Training ARIMAX on 80% Train Set...
13:08:49  INFO          Re-training ARIMAX on 90% (Train + Val) for Test Set...
13:08:55  INFO          Val  — MAE=0.


--- VALIDATION METRICS ---
Ticker        Set      MSE      MAE     MAPE     RMSE       R2       DA      TPA   V-RMSE
   VNM Validation 0.535665 0.530479 0.008646 0.731891 0.946435 0.444444 0.480000 0.731883
   FPT Validation 1.436429 0.814390 0.010315 1.198511 0.996765 0.501235 0.531414 1.187225
   MSN Validation 2.243293 1.102691 0.014878 1.497763 0.957200 0.498765 0.526042 1.496629
   VCB Validation 0.502764 0.522026 0.009161 0.709059 0.962748 0.466667 0.505348 0.708804
   VIC Validation 0.253688 0.298980 0.011868 0.503675 0.976326 0.479012 0.540390 0.503385
   HPG Validation 0.114547 0.250179 0.012632 0.338448 0.984471 0.446914 0.486559 0.338295

--- TEST METRICS ---
Ticker  Set       MSE      MAE     MAPE     RMSE       R2       DA      TPA   V-RMSE
   VNM Test  1.049594 0.688835 0.011445 1.024497 0.944659 0.431034 0.462963 1.024484
   FPT Test  3.616374 1.388160 0.013847 1.901677 0.981932 0.490148 0.505076 1.899349
   MSN Test  1.884188 0.961001 0.013095 1.372657 0.958981 0.46798